# Múltiplas fontes: a mesma busca na Presidência, na Câmara e no Senado

Pesquisa sobre legislação costuma combinar bases, porque cada uma cobre um recorte e cada buscador interpreta o termo de um jeito. Este notebook busca "doenças raras" nas três fontes legislativas do raspe, põe os resultados num formato comum e compara o que cada uma devolve. As funções vêm do script `examples/02_multiplas_fontes.py`.

**Fontes:** [Presidência da República](https://legislacao.presidencia.gov.br/), [Câmara dos Deputados](https://www.camara.leg.br/legislacao) e [Senado Federal](https://www6g.senado.leg.br/busca)

**Para rodar:** o notebook importa funções de `examples/`, então precisa de um clone do repositório e deve ser aberto de dentro da pasta `notebooks/`. No Google Colab, rode antes, numa célula: `!pip install git+https://github.com/bdcdo/raspe.git`, `!git clone https://github.com/bdcdo/raspe.git` e `%cd raspe/notebooks`.

## Preparação

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

# A pasta examples/ fica ao lado de notebooks/. Incluí-la no caminho de importação deixa o
# notebook usar as funções do script sem copiá-las.
sys.path.insert(0, str(Path.cwd().parent / "examples"))

# O nome do script começa com número, o que o comando import não aceita; importlib aceita.
multiplas = importlib.import_module("02_multiplas_fontes")

## Como cada fonte é buscada

A lista `FONTES` do script guarda, para cada fonte, o termo de busca, as páginas e quais colunas fazem o papel de título, link e texto. A sintaxe da busca muda de uma fonte para outra: a Presidência aceita aspas para pedir a expressão exata, e a Câmara responde com erro quando recebe aspas.

In [2]:
pd.DataFrame(multiplas.FONTES).drop(columns="criar")

,nome,pesquisa,paginas,coluna_titulo,coluna_link,colunas_texto
0,Presidência,"""doenças raras""","(1, 2, 3)",nome,link,[descricao]
1,Câmara,doenças raras,"(1, 2, 3, 4, 5, 6)",titulo,link,"[descricao, ementa]"
2,Senado,doenças raras,"(1, 2, 3, 4, 5, 6)",titulo,link_norma,"[descricao, trecho_descricao]"


## Coleta

`coletar_todas()` coleta uma fonte por vez. Se uma falhar, ela avisa, devolve a fonte vazia e segue com as outras. A coleta faz até 20 requisições e leva de um a dois minutos.

In [3]:
brutos = multiplas.coletar_todas(multiplas.FONTES)
for nome, dados in brutos.items():
    print(f"{nome}: {len(dados)} linhas, colunas {list(dados.columns)}")

Coletando Presidência ("doenças raras")


Baixando documentos:   0%|          | 0/3 [00:00<?, ?it/s]

Baixando documentos:  33%|███▎      | 1/3 [00:02<00:05,  2.67s/it]

Baixando documentos:  67%|██████▋   | 2/3 [00:05<00:02,  2.67s/it]

Baixando documentos: 100%|██████████| 3/3 [00:08<00:00,  2.68s/it]

...

Baixando documentos: 0it [00:00, ?it/s]

Baixando documentos: 0it [00:00, ?it/s]

Processando documentos: 0it [00:00, ?it/s]

Processando documentos: 0it [00:00, ?it/s]

  Nenhum resultado.
Presidência: 29 linhas, colunas ['nome', 'link', 'ficha', 'revogacao', 'descricao', 'termo_busca']
Câmara: 59 linhas, colunas ['link', 'titulo', 'descricao', 'ementa', 'termo_busca']
Senado: 0 linhas, colunas ['termo_busca']


Cada fonte devolve colunas próprias. Para comparar, é preciso dizer qual coluna de cada uma faz o papel de título, de link e de texto descritivo. É o que `padronizar()` faz com base em `FONTES`. Em seguida, `juntar()` empilha as três tabelas, tira resultados repetidos dentro de cada fonte e acrescenta duas colunas:

- `chave_norma`, um identificador "tipo número/ano" extraído do título, que permite reconhecer a mesma norma em fontes que escrevem o título de jeitos diferentes;
- `menciona_termo`, que diz se o título ou a ementa citam a expressão buscada.

In [4]:
resultados = multiplas.juntar(brutos, multiplas.FONTES)
print(f"{sum(len(dados) for dados in brutos.values())} linhas brutas, {len(resultados)} depois de tirar as repetidas")
resultados.groupby("fonte").head(2)

88 linhas brutas, 88 depois de tirar as repetidas


,fonte,titulo,link,texto,chave_norma,menciona_termo
0,Presidência,Lei nº 14.133 de 01 de abril de 2021Veto Parcial,https://www.planalto.gov.br/ccivil_03/_ato2019...,Lei de Licitações e Contratos Administrativos.,lei 14133/2021,False
1,Presidência,Decreto nº 10.558 de 03 de dezembro de 2020,https://www.planalto.gov.br/ccivil_03/_ato2019...,Institui o Comitê Interministerial de Doenças ...,decreto 10558/2020,True
29,Câmara,"Decreto nº 12.343, de 5 de Maio de 1943",https://www2.camara.leg.br/legin/fed/decret/19...,Ementa: Aprova o Regimento da Secretaria de Es...,decreto 12343/1943,False
30,Câmara,"Lei nº 488, de 15 de Novembro de 1948",https://www2.camara.leg.br/legin/fed/lei/1940-...,"Ementa: Dispõe sôbre pagamento de vencimento, ...",lei 488/1948,False


## Análise cruzada 1: quanto do que cada fonte devolve cita o termo

Um resultado de busca não é, por si, uma norma sobre o tema. Os buscadores procuram a expressão também no texto integral, e uma norma que trata de doenças raras num único dispositivo aparece na busca sem citar o termo no título nem na ementa. A coluna `percentual_que_cita` mede essa diferença entre o que o buscador devolve e o que trata do tema logo na ementa.

In [5]:
multiplas.resumir_por_fonte(resultados, list(brutos))

,fonte,resultados,citam_o_termo,percentual_que_cita
0,Presidência,29,6,20.7
1,Câmara,59,5,8.5
2,Senado,0,0,NaN


In [6]:
# Os resultados que citam a expressão no título ou na ementa, de todas as fontes.
resultados.loc[resultados["menciona_termo"], ["fonte", "titulo", "texto"]]

,fonte,titulo,texto
1,Presidência,Decreto nº 10.558 de 03 de dezembro de 2020,Institui o Comitê Interministerial de Doenças ...
8,Presidência,Lei nº 13.930 de 10 de dezembro de 2019,"Altera a Lei nº 10.332, de 19 de dezembro de 2..."
9,Presidência,Mensagem nº 474 de 02 de outubro de 2019Veto T...,"Projeto de Lei nº 6.566, de 2013 (no 231/12 no..."
16,Presidência,Lei nº 14.593 de 02 de junho de 2023,"Altera a Lei nº 13.693, de 10 de julho de 2018..."
19,Presidência,Decreto nº 10.916 de 28 de dezembro de 2021,"Altera o Decreto nº 10.558, de 3 de dezembro d..."
22,Presidência,Lei nº 13.693 de 10 de julho de 2018,INSTITUI O DIA NACIONAL DE DOENÇAS RARAS.INSTI...
46,Câmara,"Lei nº 13.693, de 10 de Julho de 2018",Ementa: Institui o Dia Nacional de Doenças Rar...
48,Câmara,"Lei nº 13.930, de 10 de Dezembro de 2019","Ementa: Altera a Lei nº 10.332, de 19 de dezem..."
74,Câmara,"Decreto nº 10.558, de 3 de Dezembro de 2020",Ementa: Institui o Comitê Interministerial de ...
76,Câmara,"Decreto nº 10.916, de 28 de Dezembro de 2021","Ementa: Altera o Decreto nº 10.558, de 3 de de..."


## Análise cruzada 2: normas presentes em mais de uma fonte

Com a chave da norma, dá para ver quais normas as fontes têm em comum. Uma norma que só uma fonte devolve não é necessariamente ausente das outras: ela pode estar lá, mas fora das páginas coletadas ou fora do critério do buscador.

In [7]:
em_comum = multiplas.normas_em_comum(resultados)
print(f"{len(em_comum)} normas aparecem em mais de uma fonte")
em_comum.head(10)

19 normas aparecem em mais de uma fonte


,chave_norma,fontes,n_fontes
2,decreto 10177/2019,"Câmara, Presidência",2
3,decreto 10558/2020,"Câmara, Presidência",2
4,decreto 10916/2021,"Câmara, Presidência",2
5,decreto 10987/2022,"Câmara, Presidência",2
10,decreto 12411/2025,"Câmara, Presidência",2
11,decreto 12651/2025,"Câmara, Presidência",2
12,decreto 12955/2026,"Câmara, Presidência",2
36,decreto 9673/2019,"Câmara, Presidência",2
44,lei 12663/2012,"Câmara, Presidência",2
48,lei 13693/2018,"Câmara, Presidência",2


In [8]:
# A mesma norma nas duas fontes: títulos escritos de jeitos diferentes, links diferentes.
exemplo = em_comum["chave_norma"].iloc[0]
resultados.loc[resultados["chave_norma"] == exemplo, ["fonte", "titulo", "link"]]

,fonte,titulo,link
5,Presidência,Decreto nº 10.177 de 16 de dezembro de 2019,https://www.planalto.gov.br/ccivil_03/_ato2019...
70,Câmara,"Decreto nº 10.177, de 16 de Dezembro de 2019",https://www2.camara.leg.br/legin/fed/decret/20...


## Análise cruzada 3: o período que cada fonte cobre

O ano da norma, tirado da chave, mostra de que época vem o que cada fonte devolveu. A Câmara ordena os resultados do mais antigo para o mais recente, então páginas coletadas a mais ou a menos mudam bastante o retrato dela.

In [9]:
ano = resultados["chave_norma"].str.extract(r"/(\d{4})$")[0].astype("Int64")
decada = (ano // 10 * 10).rename("década")
pd.crosstab(decada, resultados["fonte"], margins=True, margins_name="Total")

fonte,Câmara,Presidência,Total
década,,,
1880,2,0,2
1910,4,0,4
1920,6,0,6
1930,8,0,8
1940,2,0,2
1950,4,0,4
1960,2,0,2
1970,3,0,3
1980,1,1,2


## Quando uma fonte volta vazia

Uma fonte com zero linhas no resumo pode ter falhado na coleta (site fora do ar, página de verificação contra robôs) ou não ter achado nada. O código trata os dois casos do mesmo jeito para que a análise siga, mas antes de concluir que uma base não tem normas sobre o tema vale abrir a busca no site e conferir.

In [10]:
[nome for nome, dados in brutos.items() if dados.empty]

['Senado']

## Para ir além

- O script `examples/02_multiplas_fontes.py` faz a mesma coleta e salva as três tabelas em `examples/datasets/`.
- Para acrescentar uma fonte, inclua um dicionário em `multiplas.FONTES` com o raspador, o termo, as páginas e o papel de cada coluna. Nenhuma outra função precisa mudar.